In [ ]:
import os
import subprocess
import tempfile
import pandas as pd

from settings import Settings

settings = Settings()

PROJECT_DIR = settings.project_dir   # folder with dbt_project.yml

def mf_query(metrics, group_by=None, where=None, order=None,
             limit=None, start_time=None, end_time=None):
    with tempfile.TemporaryDirectory() as tmp:
        out = os.path.join(tmp, "out.csv")
        cmd = ["uv", "run", "mf", "query",
               "--metrics", ",".join(metrics), "--csv", out]
        if group_by:   cmd += ["--group-by", ",".join(group_by)]
        if where:      cmd += ["--where", where]
        if order:      cmd += ["--order", ",".join(order)]
        if limit:      cmd += ["--limit", str(limit)]
        if start_time: cmd += ["--start-time", start_time]
        if end_time:   cmd += ["--end-time", end_time]

        try:
            subprocess.run(
                cmd, cwd=PROJECT_DIR, check=True, capture_output=True, text=True,
                env={**os.environ, "DBT_PROFILES_DIR": PROJECT_DIR},
            )
        except subprocess.CalledProcessError as e:
            raise RuntimeError(e.stderr or e.stdout) from e
        return pd.read_csv(out)

Use it

In [ ]:
df = mf_query(
    metrics=["order_total", "order_count"],
    group_by=["metric_time__month", "store"],
    order=["metric_time__month", "store"],
    limit=50,
)
df

A filter looks like this:

In [ ]:
df = mf_query(
    metrics=["order_total", "order_count"],
    group_by=["metric_time__month"],
    where="{{ Entity('store') }} = 'Philadelphia'",
    order=["metric_time__month"],
    limit=50,
)
df

Since the result is a DataFrame, you can plot it straight away

In [ ]:
import matplotlib.pyplot as plt

df["metric_time__month"] = pd.to_datetime(df["metric_time__month"])
df.plot(x="metric_time__month", y="order_total")